# Filter Nepali YouTube Comments for a Benchmark Dataset

Reads a CSV with a `comment` column (plus any other columns) and produces a cleaned, **representative** benchmark dataset.

Every comment is either **kept** or **dropped** — nothing is flagged for review. The dataset is meant to reflect how Nepali speakers actually write online, so Devanagari, Roman Nepali, code-mixed, long, and English comments are all kept.

**Drop only these:**
- No meaningful content (emoji-only / punctuation-only): `😂😂😂`, `👍👍`
- Extremely short comments that can't be classified: `wow`, `nice`, `ok`
- Spam: `Subscribe my channel`, `Visit my website`
- Exact duplicates
- Comments that are only usernames, hashtags, or links

Everything else is kept. Metadata columns (`script`, `language`, `emoji_count`, `word_count`, `source`) are still added so researchers can slice results by Devanagari / Roman Nepali / code-mixed / emoji-heavy later.

## 1. Configuration

In [ ]:
from pathlib import Path

INPUT_CSV = "comments.csv"            # path to your source CSV
OUTPUT_CSV = "comments_filtered.csv"  # cleaned dataset (kept rows only)

COMMENT_COL = "comment"               # the column to filter on
SOURCE_VALUE = "youtube"              # value written into the `source` metadata field

MIN_TOKENS = 2          # comments with fewer real word-tokens are dropped...
SHORT_WHITELIST = set() # ...unless the lowercased text is a valid label here, e.g. {"ok"}
DROP_DUPLICATES = True  # drop exact duplicate texts (keeps first occurrence)

## 2. Detection helpers

Pure functions, no pandas. HTML is unescaped and tags stripped first so anchor/link-only comments are caught correctly.

In [ ]:
import re
import html
import unicodedata

DEVANAGARI_RE = re.compile(r"[\u0900-\u097F]")
LATIN_RE = re.compile(r"[A-Za-z]")
HTML_TAG_RE = re.compile(r"<[^>]+>")
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#\w+")
WORD_RE = re.compile(r"[\w\u0900-\u097F]+", re.UNICODE)

SPAM_PATTERNS = [
    r"subscribe\s+(to\s+)?my\s+channel",
    r"visit\s+my\s+(website|channel|page)",
    r"check\s+out\s+my\s+channel",
    r"follow\s+me\s+on",
    r"sub\s*4\s*sub",
    r"promo\s*code",
    r"click\s+(the\s+)?link",
]
SPAM_RE = re.compile("|".join(SPAM_PATTERNS), re.IGNORECASE)

# Roman-Nepali function words: presence marks a Latin-script comment as Nepali / code-mixed.
ROMAN_NEPALI_HINTS = {
    "cha", "chha", "xa", "ramro", "naramro", "ho", "hoina", "timi", "hami",
    "malai", "timro", "mero", "hamro", "dai", "bhai", "didi", "bhayo", "garyo",
    "garnu", "lastai", "dherai", "ekdam", "sabai", "yo", "tyo", "kasto", "kati",
    "raicha", "rahecha", "vaye", "bhaye", "huncha", "chaina", "thiyo", "barema",
    "ani", "tara", "pani", "matra", "nai", "hai", "hola", "jasto", "haldinu",
}


def clean_html(text: str) -> str:
    """Unescape HTML entities (&#39; etc.) and strip tags (<a href=...>)."""
    return HTML_TAG_RE.sub(" ", html.unescape(text))


def count_emojis(text: str) -> int:
    n = 0
    for ch in text:
        cp = ord(ch)
        if (
            0x1F300 <= cp <= 0x1FAFF
            or 0x2600 <= cp <= 0x27BF
            or 0x1F000 <= cp <= 0x1F0FF
            or cp in (0x2764, 0x2B50, 0x2728)
            or unicodedata.category(ch) == "So"
        ):
            n += 1
    return n


def strip_emojis(text: str) -> str:
    return "".join(
        ch for ch in text
        if not (
            0x1F000 <= ord(ch) <= 0x1FAFF
            or 0x2600 <= ord(ch) <= 0x27BF
            or unicodedata.category(ch) == "So"
        )
    )


def word_tokens(text: str):
    """Word tokens with URLs / mentions / hashtags removed."""
    cleaned = HASHTAG_RE.sub(" ", MENTION_RE.sub(" ", URL_RE.sub(" ", text)))
    return WORD_RE.findall(cleaned)


def classify_script(text: str) -> str:
    """Return one of: devanagari, roman_nepali, code_mixed, latin, other, empty."""
    has_dev = bool(DEVANAGARI_RE.search(text))
    has_lat = bool(LATIN_RE.search(text))
    toks = [t.lower() for t in word_tokens(text)]
    has_roman_hint = any(t in ROMAN_NEPALI_HINTS for t in toks)

    if has_dev and has_lat:
        return "code_mixed"
    if has_dev:
        return "devanagari"
    if has_lat:
        return "roman_nepali" if has_roman_hint else "latin"
    if strip_emojis(text).strip() == "":
        return "empty"
    return "other"


def classify_language(script: str) -> str:
    return {
        "devanagari": "nepali",
        "roman_nepali": "nepali",
        "code_mixed": "nepali_english",
        "latin": "english",
        "empty": "none",
        "other": "other",
    }.get(script, "other")

In [ ]:
def evaluate(text):
    """Classify one comment. Returns decision ('keep' | 'drop'), a reason, and metadata."""
    raw = "" if text is None else str(text)
    text = clean_html(raw).strip()
    script = classify_script(text)
    emoji_n = count_emojis(text)
    toks = word_tokens(text)
    n_words = len(toks)
    text_no_emoji = strip_emojis(text).strip()

    meta = {
        "script": script,
        "language": classify_language(script),
        "emoji_count": emoji_n,
        "word_count": n_words,
        "source": SOURCE_VALUE,
    }

    def out(decision, reason):
        return {"decision": decision, "reason": reason, **meta}

    if text == "":
        return out("drop", "empty")

    # Only emojis / punctuation — no words at all.
    if n_words == 0:
        return out("drop", "no_meaningful_content")

    # Only URLs, mentions, or hashtags.
    stripped_meta = HASHTAG_RE.sub("", MENTION_RE.sub("", URL_RE.sub("", text))).strip()
    if strip_emojis(stripped_meta).strip() == "":
        return out("drop", "only_links_mentions_or_hashtags")

    # Spam.
    if SPAM_RE.search(text):
        return out("drop", "spam")

    # Too short to classify — unless whitelisted as a valid label.
    if n_words < MIN_TOKENS and text_no_emoji.lower() not in SHORT_WHITELIST:
        return out("drop", "too_short")

    # Everything else is kept: Devanagari, Roman Nepali, code-mixed, long, and English.
    return out("keep", "ok")


for sample in [
    "यो भिडियो धेरै राम्रो छ।",
    "yo video dherai ramro cha",
    "Video ramro cha but editing could be better.",
    "Yo ta lastai ramro 🔥",
    "so amazingly said , well said , well truth !",
    "Come to europe",
    "😂😂😂😂",
    "nice",
    "Subscribe my channel",
    '<a href="https://www.youtube.com/live/UgeYdADx">link</a>',
    "It&#39;s a time to show such authentic case studies",
]:
    r = evaluate(sample)
    print(f"{r['decision']:5} | {r['reason']:30} | {r['script']:12} | {sample[:50]}")

## 3. Load the CSV and apply the filter

In [ ]:
import pandas as pd

df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df):,} rows, columns: {list(df.columns)}")

if COMMENT_COL not in df.columns:
    raise KeyError(
        f"Column '{COMMENT_COL}' not found. Available: {list(df.columns)}. "
        "Update COMMENT_COL in the config cell."
    )

evaluated = df[COMMENT_COL].apply(evaluate).apply(pd.Series)
df = pd.concat([df, evaluated], axis=1)

# Mark exact duplicates (keep first), after evaluate so a kept copy isn't lost to a dropped one.
if DROP_DUPLICATES:
    norm = df[COMMENT_COL].astype(str).str.strip().str.lower()
    is_dup = norm.duplicated(keep="first")
    df.loc[is_dup & (df["decision"] == "keep"), ["decision", "reason"]] = ["drop", "duplicate"]

df["decision"].value_counts()

## 4. Inspect what is being dropped

Eyeball this before trusting the output — the thresholds and spam lexicon are heuristics.

In [ ]:
print("Breakdown by reason:")
print(df.groupby(["decision", "reason"]).size().to_string())

print("\nScript distribution among KEPT rows:")
print(df.loc[df["decision"] == "keep", "script"].value_counts().to_string())

print("\nSample of dropped comments:")
display(df.loc[df["decision"] == "drop", [COMMENT_COL, "reason", "script"]].head(30))

## 5. Write output

`OUTPUT_CSV` contains the kept rows with the metadata columns, ready for labelling.

In [ ]:
keep_cols = list(df.columns.drop(["decision", "reason"]))
kept = df[df["decision"] == "keep"][keep_cols].reset_index(drop=True)

kept.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"Kept    : {len(kept):,} rows -> {OUTPUT_CSV}")
print(f"Dropped : {(df['decision'] == 'drop').sum():,} rows")